# Task 2A — Policy-Grounded Financial Compliance Triage
Prepare an auditable synthetic dataset from fictional internal policies and scenarios.
The target is JSON with `category`, `risk_level`, `recommended_action`, and `rationale`.
Only the supplied policy governs the answer. This is an educational internal-policy
task, **not legal advice**; no jurisdiction-specific law or real customer data is used.

Teacher: **openai/gpt-oss-120b via Groq**. Planned student:
**Qwen/Qwen2.5-3B-Instruct**. They are distinct models. Task 2A prepares data only;
student loading, training and evaluation are reserved for Tasks 2B/2C.

Run cells in order. Generation uses real API responses, never invented outputs.
The default run requests 160 candidates in 16 batches of 10 and requires at least
120 cleaned records before creating splits. With 60-second free-tier pacing,
allow roughly 16 minutes plus generation/retry time. Retained size is measured,
not guaranteed. Reopening a completed saved dataset reuses it without teacher calls.

## 1. Repository/dependency setup
Open from your GitHub repository in Colab. If no checkout is present, supply its
public GitHub URL. Install the existing requirements; no training dependencies are
added. Locally, start Jupyter from the repository root after installing requirements.

In [ ]:
import os
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

try:
    IN_COLAB = find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "task2_genai/prompts.py").is_file()
    ),
    None,
)
if repo_root is None:
    repo_url = input("Public GitHub repository HTTPS URL: ").strip()
    if not repo_url.startswith("https://github.com/"):
        raise ValueError("Supply the public GitHub repository URL")
    checkout = Path.cwd() / "CDAZZDEV-MLE-Poshitha-Malagala"
    if not checkout.exists():
        subprocess.run(["git", "clone", repo_url, str(checkout)], check=True)
    repo_root = checkout.resolve()
    if not (repo_root / "task2_genai/prompts.py").is_file():
        raise RuntimeError("Use a checkout containing the Task 2A implementation")
if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(repo_root / "requirements.txt"),
        ],
        check=True,
    )
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("Repository:", repo_root)

## 2. Full teacher system prompt
The production prompt is saved in `task2_genai/teacher_system_prompt.txt`.

```text
You are generating a dataset for policy-grounded financial compliance triage.
Generate realistic but entirely fictional scenarios and target answers. These are
internal synthetic policies, not jurisdiction-specific law. Use ONLY the supplied
policy excerpt for each assignment; never invoke external regulations or provide
legal advice. Do not invent extra policy requirements or thresholds.

Return ONE JSON object with an "examples" array and no surrounding commentary.
Produce exactly one example per supplied assignment. Each object must have exactly:
id, topic, policy_excerpt, scenario, category, risk_level, recommended_action,
rationale. Copy the assignment's id, topic, category and policy_excerpt exactly.
risk_level must be "low", "medium" or "high". Construct a scenario consistent with
the assigned target risk; the final risk/action must follow the supplied policy.
The recommended_action must be actionable and follow the applicable internal rule.
The rationale must explicitly refer to the supplied policy and connect its trigger
to the scenario facts. Never add facts that are absent from the scenario or policy.
Do not mention that data is synthetic or generated in the answer fields.

Use anonymous roles such as "a retail customer" or "a small business", not personal
or company names. Do not include real customer data, names, account numbers, emails,
phone numbers, addresses, government identifiers or other PII. Use generic location
labels from the internal policy rather than naming countries or real restricted lists.
Transaction amounts may be fictional; size alone is not a risk trigger unless the
supplied policy explicitly says so.

Vary situation, sentence structure, evidence, customer type, transaction size,
scenario length, difficulty and ambiguity using the assignment hints. Include
borderline cases and uncertainty: explain when missing evidence requires review
rather than assuming wrongdoing. For ambiguous cases, provide enough facts for the
policy's uncertainty rule to apply. Avoid stereotyped, repeated or lightly renamed
templates, including templates from the avoidance examples. Do not simply paraphrase
the policy into a scenario. Give concrete operational facts and meaningful contrasts
between confirmed concerns, unresolved concerns and verified clearances.

Keep scenarios within the requested word range, actions concise, and rationales
approximately 25-55 words. Keep the exact supplied policy text even if another wording
would be shorter. All fields must be non-empty strings. Produce valid JSON only.
```

In [ ]:
from task2_genai.prompts import TEACHER_SYSTEM_PROMPT
from task2_genai.schemas import STUDENT_MODEL, TEACHER_MODEL

print("Assessment teacher:", TEACHER_MODEL)
print("Planned student (not loaded):", STUDENT_MODEL)
print(TEACHER_SYSTEM_PROMPT)

## 3. Synthetic policy taxonomy
Fourteen topics each define low/medium/high triggers and actions. Two equivalent
excerpt styles vary wording without changing policy semantics. Every example must
echo its assigned excerpt exactly; neither scenarios nor answers use real laws.

In [ ]:
import pandas as pd
from IPython.display import JSON, display

from task2_genai.policies import POLICIES

display(
    pd.DataFrame(
        [
            {
                "topic": policy.topic,
                "category": policy.category,
                "policy_excerpt": policy.excerpt(),
                "alternate_wording": policy.excerpt(1),
            }
            for policy in POLICIES
        ]
    )
)

## 4. Generation configuration and secure Groq setup
Set `GROQ_API_KEY` in the environment or Colab Secrets and allow notebook access.
`GROQ_MODEL` is configurable; use `openai/gpt-oss-120b` for this assessment.
The key is never displayed. Default batch size 10 and low supported reasoning
effort keep requests compact; provider limits still apply. Increase completion
budget if using batches of 20 on an account with sufficient quota.

Pacing is at the Task 2 transport boundary only; disable it on higher quotas.
Retryable errors use at most three transport attempts with backoff/Retry-After.
Invalid records are rejected locally, not silently repaired or relabeled.

In [ ]:
import getpass
import json
import logging

from task2_genai.client import GroqTeacherClient, TeacherError
from task2_genai.schemas import GenerationConfig, PolicyExample

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
config = GenerationConfig(
    candidate_count=160, batch_size=10, minimum_clean=120, seed=42
)
RUN_GENERATION = True
FREE_TIER_PACING = True
DATA_DIR = repo_root / "task2_genai" / "data"
RAW_PATH = DATA_DIR / "raw_teacher_examples.jsonl"
CLEAN_PATH = DATA_DIR / "cleaned_examples.jsonl"
SUMMARY_PATH = DATA_DIR / "analysis_summary.json"
teacher_model = os.environ.get("GROQ_MODEL") or TEACHER_MODEL
client = None
if RUN_GENERATION and not CLEAN_PATH.exists() and not RAW_PATH.exists():
    if not os.environ.get("GROQ_API_KEY") and IN_COLAB:
        try:
            from google.colab import userdata

            secret = userdata.get("GROQ_API_KEY")
            if secret:
                os.environ["GROQ_API_KEY"] = secret
            secret = None
            configured_model = userdata.get("GROQ_MODEL")
            if configured_model:
                teacher_model = configured_model
        except Exception:
            print("Colab Secret unavailable; use hidden key input if needed.")
    if not os.environ.get("GROQ_API_KEY"):
        key = getpass.getpass("Groq API key (hidden; blank skips live generation): ")
        if key:
            os.environ["GROQ_API_KEY"] = key
        key = None
    try:
        client = GroqTeacherClient(
            model=teacher_model,
            max_completion_tokens=4500,
            min_interval_seconds=60 if FREE_TIER_PACING else 0,
        )
    except TeacherError as error:
        print(str(error), "No data will be substituted.")
print("Teacher model:", teacher_model, "| Student:", STUDENT_MODEL)
if teacher_model == STUDENT_MODEL:
    raise ValueError("Teacher and student must differ")
display(JSON(config.model_dump()))

## 5. Batched teacher generation
The balanced plan controls topic, target risk, anonymous customer type, amount band,
scenario length and ambiguity. Five recent accepted scenarios discourage repeated
templates. Raw batch envelopes retain exact responses and assignment/prompt provenance.
Each completed batch is checkpointed. Partial failures reduce coverage; missing
examples are never synthesized locally. Review the logs and raw file if interrupted.

In [ ]:
from task2_genai.generation import build_assignments, generate_examples

generation = None
clean_examples = []
summary = {}
reusing_saved = CLEAN_PATH.exists()
if reusing_saved:
    clean_examples = [
        PolicyExample.model_validate(json.loads(line))
        for line in CLEAN_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if SUMMARY_PATH.exists():
        summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    print("Reusing saved clean records; zero teacher calls:", len(clean_examples))
elif RAW_PATH.exists():
    print(
        "A raw run already exists without a clean dataset. Inspect it before regenerating;"
    )
    print("choose a new DATA_DIR in configuration to preserve the existing raw audit.")
elif client is not None and RUN_GENERATION:
    display(
        pd.DataFrame(build_assignments(config))
        .groupby(["topic", "target_risk"])
        .size()
        .rename("planned")
    )
    generation = generate_examples(client, config, raw_path=RAW_PATH)
    print("Generation completed. Raw audit:", RAW_PATH)
else:
    print("Generation skipped; no examples fabricated.")

## 6. Local validation summary
Pydantic validates risks, required text, lengths and taxonomy. Generation also checks
IDs, exact supplied policy and assigned risk. Schema/assignment failures are rejected;
valid siblings survive. Structural checks do not prove semantic correctness: manually
review policy/action/rationale consistency before submitting or training.

In [ ]:
if generation is not None:
    summary = {
        "configuration": config.model_dump(),
        "teacher_model": teacher_model,
        "validation": {
            "planned_candidates": config.candidate_count,
            "returned_items": generation.returned_count,
            "accepted_before_dedup": len(generation.examples),
            "rejected_records": len(generation.rejected),
            "missing_requested_ids": generation.missing_count,
            "failed_batches": generation.failed_batches,
        },
        "rejections": generation.rejected,
    }
    display(JSON(summary["validation"]))
    if generation.rejected:
        display(pd.DataFrame(generation.rejected))
elif summary:
    display(JSON(summary.get("validation", {})))
else:
    print("No validation results yet; run live generation with credentials.")

## 7. Exact and near-duplicate filtering
Normalize case/Unicode/punctuation for exact scenario comparisons. Token sequence
similarity at **0.88 or above** flags near duplicates; retain the first record and
record removals/similarity. This lexical heuristic is intentionally lightweight:
inspect borderline pairs and distributions rather than claiming semantic deduplication.
Filtering is global before splitting, including across different policy topics.

In [ ]:
from task2_genai.dataset import deduplicate_examples, examples_frame, write_jsonl

if generation is not None:
    deduplication = deduplicate_examples(generation.examples)
    clean_examples = deduplication.examples
    summary["deduplication"] = deduplication.statistics
    summary["duplicate_removals"] = deduplication.removed
    write_jsonl(CLEAN_PATH, (example.model_dump() for example in clean_examples))
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
if summary.get("deduplication"):
    display(JSON(summary["deduplication"]))
    if summary.get("duplicate_removals"):
        display(pd.DataFrame(summary["duplicate_removals"]))
else:
    print("No deduplication results yet.")
frame = examples_frame(clean_examples)

## 8. Topic/category and risk-level diversity
Report measured distributions, not just planned coverage. No custom chart colors are specified.

In [ ]:
import matplotlib.pyplot as plt

if not frame.empty:
    display(frame["topic"].value_counts().rename("topic_count"))
    display(frame["category"].value_counts().rename("category_count"))
    display(frame["risk_level"].value_counts().rename("risk_count"))
    display(pd.crosstab(frame["topic"], frame["risk_level"]))
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for column, axis in zip(["topic", "category", "risk_level"], axes):
        frame[column].value_counts().plot.bar(ax=axis, title=column, ylabel="Examples")
    plt.tight_layout()
    plt.show()
else:
    print("No clean records available to plot.")

## 9. Input/scenario length diversity
Lengths are words in the policy+scenario and scenario alone. Token lengths will be measured with the student tokenizer in Task 2B.

In [ ]:
if not frame.empty:
    display(frame[["input_words", "scenario_words"]].describe())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    frame["input_words"].plot.hist(ax=axes[0], bins=15, title="Input length (words)")
    frame["scenario_words"].plot.hist(
        ax=axes[1], bins=15, title="Scenario length (words)"
    )
    axes[0].set_xlabel("Words")
    axes[1].set_xlabel("Words")
    plt.tight_layout()
    plt.show()

## 10. Final retained size
Do not claim the dataset is ready when fewer than 120 clean examples remain. Preserve actual partial data; inspect rejection/diversity results and make a separate real generation run if needed.

In [ ]:
dataset_ready = len(clean_examples) >= config.minimum_clean
print("Final clean examples:", len(clean_examples))
print("Minimum required:", config.minimum_clean, "| Ready to split:", dataset_ready)
if not dataset_ready:
    print("Splitting is blocked; no missing examples will be fabricated.")

## 11. Reproducible 80/10/10 split
Fixed seed 42. Use topic+risk strata when feasible, then topic, then risk; small
holdouts cannot represent all 42 topic/risk cells. Proportional largest-remainder
allocation gives exact rounded sizes. Global deduplication precedes the split.
Saved IDs/hashes make the test set auditable. Existing split files are never
overwritten; keep the test set untouched until Task 2C evaluation.

In [ ]:
import hashlib

from task2_genai.dataset import save_splits, split_examples

manifest_path = DATA_DIR / "split_manifest.json"
if dataset_ready and manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected_hash in manifest["sha256"].items():
        assert (
            hashlib.sha256((DATA_DIR / f"{name}.jsonl").read_bytes()).hexdigest()
            == expected_hash
        )
    print("Existing split hashes verified; test set unchanged.")
    print(
        "Exact counts:",
        manifest["counts"],
        "| Stratification:",
        manifest["stratification"],
    )
elif dataset_ready:
    split = split_examples(clean_examples, seed=config.seed)
    try:
        paths = save_splits(DATA_DIR, split)
        print("Exact counts:", split.counts, "| Stratification:", split.stratification)
        display(JSON(paths))
    except FileExistsError as error:
        print(str(error), "Inspect existing artifacts before continuing.")
else:
    print("No splits generated: minimum clean size not reached.")

## 12. Sample chat-format records
The assistant contains only the four target fields. Metadata IDs are not fed to
the student. Later use Qwen's `tokenizer.apply_chat_template`; no tokenizer, weights
or ChatML strings are loaded/generated in Task 2A.

In [ ]:
from task2_genai.dataset import chat_record

for example in clean_examples[:2]:
    display(JSON(chat_record(example)))
if not clean_examples:
    print("No real teacher examples available to display.")

## 13. Saved artifact paths
Download the real files from Colab's Files panel or copy them to your repository
after reviewing them. Raw JSONL holds batch envelopes; cleaned JSONL holds one
validated example per line. Split JSONL holds chat messages. Failed raw envelopes
contain safe categories and no invented replacements.

In [ ]:
artifact_names = [
    "raw_teacher_examples.jsonl",
    "cleaned_examples.jsonl",
    "analysis_summary.json",
    "train.jsonl",
    "validation.jsonl",
    "test.jsonl",
    "split_manifest.json",
]
display(
    pd.DataFrame(
        [
            {
                "artifact": name,
                "path": str(DATA_DIR / name),
                "exists": (DATA_DIR / name).exists(),
                "bytes": (DATA_DIR / name).stat().st_size
                if (DATA_DIR / name).exists()
                else None,
            }
            for name in artifact_names
        ]
    )
)

## 14. Limitations and next phases
Pydantic proves structure, not the correctness of teacher reasoning. Review a
stratified sample and all flagged/rejected records for grounding, realism and PII
before submission. Lexical deduplication can miss paraphrases or remove similar
legitimate cases; inspect its removals and resulting coverage. Teacher output bias
can carry into the student. This small dataset is not a production compliance system
and must not be used as legal advice.

Groq quotas/network availability may reduce yield; the notebook reports actual counts
and does not invent examples. Commit/download actual generated datasets and the
executed notebook as evidence. Qwen's model card specifies a Qwen research license;
review its terms before Task 2B use. Task 2B training/baseline comparisons and Task 2C
held-out evaluation are pending. Tasks 1 and 3 are unchanged.